# 🏛️ MonuVista — Agente de Chat sobre PDF

Este notebook cria um **agente de IA** que responde **apenas** com base no conteúdo de um ficheiro PDF fornecido por ti.

## Como usar
1. Executa a célula de instalação de dependências (só uma vez).
2. Define o teu `OPENAI_API_KEY` na célula de configuração.
3. Coloca o caminho para o teu ficheiro PDF na variável `PDF_PATH`.
4. Executa todas as células por ordem.
5. Na célula de chat, escreve a tua pergunta e prime **Enter** ou clica em **Enviar**.

> **Nota:** O agente recusa responder a perguntas cujo conteúdo não esteja presente no PDF.

In [ ]:
# ── 1. Instalar dependências ────────────────────────────────────────────────
# Execute esta célula apenas uma vez (ou quando configurar o ambiente pela primeira vez).
%pip install -q \
    langchain \
    langchain-community \
    langchain-openai \
    openai \
    pypdf \
    faiss-cpu \
    ipywidgets

print("✅ Dependências instaladas.")

In [ ]:
# ── 2. Configuração ─────────────────────────────────────────────────────────
import os

# Substitui pelo teu API Key do OpenAI (https://platform.openai.com/api-keys)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "sk-...")  # ou escreve directamente a key aqui

# Caminho para o teu ficheiro PDF
PDF_PATH = "meu_documento.pdf"  # ← altera este caminho

# Modelo OpenAI a usar
LLM_MODEL = "gpt-4o-mini"  # pode ser "gpt-3.5-turbo", "gpt-4", etc.

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print(f"📄 PDF: {PDF_PATH}")
print(f"🤖 Modelo: {LLM_MODEL}")

In [ ]:
# ── 3. Carregar e indexar o PDF ─────────────────────────────────────────────
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

print("📖 A carregar o PDF...")
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()
print(f"   → {len(pages)} página(s) carregada(s).")

print("✂️  A dividir em fragmentos...")
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)
docs = splitter.split_documents(pages)
print(f"   → {len(docs)} fragmento(s) criado(s).")

print("🔢 A criar embeddings e índice FAISS (pode demorar um momento)...")
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
print("✅ Índice criado com sucesso.")

In [ ]:
# ── 4. Criar o agente de QA restrito ao PDF ─────────────────────────────────
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate

# Prompt que instrui o modelo a responder APENAS com base no contexto do PDF
QA_PROMPT_TEMPLATE = """\
Usa APENAS as informações contidas no contexto abaixo para responder à pergunta.
Se a resposta não estiver no contexto, responde exactamente com:
"Não encontrei essa informação no documento."
Não inventes nem uses conhecimento externo.

Contexto:
{context}

Pergunta: {question}
Resposta:"""

QA_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=QA_PROMPT_TEMPLATE,
)

llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer",
)

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    combine_docs_chain_kwargs={"prompt": QA_PROMPT},
    return_source_documents=True,
    output_key="answer",
)

print("🤖 Agente pronto. Passa para a célula de chat!")

In [ ]:
# ── 5. Interface de Chat Interativa ─────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display, HTML

# ── Histórico de mensagens
chat_history_html = []

def render_chat():
    """Renderiza o histórico de chat na área de output."""
    output_area.clear_output(wait=True)
    with output_area:
        display(HTML("".join(chat_history_html)))

def add_message(role: str, text: str, sources: list | None = None):
    """Adiciona uma mensagem ao histórico visual."""
    if role == "user":
        bubble_style = (
            "background:#0078d4;color:white;border-radius:18px 18px 4px 18px;"
            "padding:10px 15px;margin:6px 0 6px 30%;max-width:65%;float:right;"
            "clear:both;word-wrap:break-word;"
        )
        label = "🧑 Tu"
    else:
        bubble_style = (
            "background:#f0f0f0;color:#222;border-radius:18px 18px 18px 4px;"
            "padding:10px 15px;margin:6px 30% 6px 0;max-width:65%;float:left;"
            "clear:both;word-wrap:break-word;"
        )
        label = "🤖 Agente"

    src_html = ""
    if sources:
        pages_used = sorted({doc.metadata.get('page') + 1 for doc in sources
                              if isinstance(doc.metadata.get('page'), int)})
        if pages_used:
            pages_str = ", ".join(str(p) for p in pages_used)
            src_html = (
                f"<div style='font-size:0.75em;color:#888;margin-top:4px;'>"
                f"📄 Fonte: página(s) {pages_str}</div>"
            )

    chat_history_html.append(
        f"<div style='overflow:hidden;'>"
        f"<div style='{bubble_style}'>"
        f"<strong>{label}</strong><br>{text}"
        f"{src_html}"
        f"</div></div>"
    )
    render_chat()


def on_send(button_or_event=None):
    """Callback quando o utilizador envia uma mensagem."""
    question = input_box.value.strip()
    if not question:
        return
    input_box.value = ""
    send_button.disabled = True
    input_box.disabled = True

    add_message("user", question)

    try:
        result = qa_chain.invoke({"question": question})
        answer = result.get("answer", "Sem resposta.")
        sources = result.get("source_documents", [])
        add_message("agent", answer, sources)
    except Exception as exc:
        add_message("agent", f"❌ Erro: {exc}")
    finally:
        send_button.disabled = False
        input_box.disabled = False


# ── Widgets
title = widgets.HTML(
    "<h3 style='margin:0 0 8px 0;font-family:sans-serif;'>💬 Chat com o PDF</h3>"
)
output_area = widgets.Output(
    layout=widgets.Layout(
        height="420px",
        overflow_y="auto",
        border="1px solid #ddd",
        padding="8px",
        background_color="white",
    )
)
input_box = widgets.Text(
    placeholder="Escreve a tua pergunta aqui...",
    layout=widgets.Layout(width="85%"),
)
send_button = widgets.Button(
    description="Enviar",
    button_style="primary",
    layout=widgets.Layout(width="13%"),
)
clear_button = widgets.Button(
    description="🗑 Limpar",
    button_style="warning",
    layout=widgets.Layout(width="10%"),
)

def on_clear(b):
    """Limpa o histórico de chat e a memória do agente."""
    chat_history_html.clear()
    memory.clear()
    output_area.clear_output()

send_button.on_click(on_send)
input_box.on_submit(on_send)
clear_button.on_click(on_clear)

input_row = widgets.HBox(
    [input_box, send_button, clear_button],
    layout=widgets.Layout(margin="6px 0 0 0"),
)

ui = widgets.VBox(
    [title, output_area, input_row],
    layout=widgets.Layout(width="100%", padding="10px"),
)

# ── Mensagem de boas-vindas
add_message(
    "agent",
    f"Olá! Estou pronto para responder às tuas perguntas sobre o ficheiro "
    f"<strong>{os.path.basename(PDF_PATH)}</strong>. "
    "Só consigo responder com base no conteúdo desse documento.",
)

display(ui)